# MediFlow 스마트폰 얼굴 · 고정 최종 Source Test

**GPU 선택 → 모두 실행**. 기존 공통 실행 ZIP과 Drive에 저장된 1~5차 모델을 재사용합니다.
**새 ZIP 업로드·복호화 키 입력·새 학습은 없습니다.**

- 기존 원본 Test 400장, 건선·아토피·여드름·정상·주사 순서, PMG·B0·256을 유지합니다.
- 적응 전·1·2·3·4·추가5차의 6개 모델 × 원본·1차 조명·2차 약한 확장·4차 강한 확장·새 공동 근접 변환의 5조건, 총 30개 비교입니다.
- 모든 모델에 같은 픽셀 입력·seed 1707을 씁니다. 변환 조건은 평가 시 100% 적용합니다.
- **Validation에서 고른 4차를 미리 고정**합니다. Test 결과로 모델·epoch·강도를 다시 고르지 않습니다.
- 사진 원본·학습 가중치를 바꾸지 않습니다. 실제 휴대폰 사진은 읽지 않고 Target Accuracy를 계산하지 않습니다.
- 원본과 변환별 Accuracy·Macro-F1·클래스 Recall·혼동행렬·개별 예측·비교 그래프가 나옵니다.
- 모델·조건·배치 진행률을 표시합니다. 중단되면 같은 노트북을 모두 실행하면 검증된 완료 조건을 재사용합니다.

결과는 `MyDrive/MediFlow_smartphone_runs/test_ad2338eaa932`에 저장되며
`MediFlow_Smartphone_Final_Test_Results_20261112.zip`을 Drive에 만들고 자동 다운로드합니다. 결과 ZIP에는 모델·사진·키가 들어가지 않습니다.
Source의 사람/병변/세션 식별 누락과 과거 체크포인트의 Test 노출 미확인 한계를 유지한 평가입니다.

In [ ]:
from google.colab import drive
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, os, subprocess, sys, zipfile
drive.mount("/content/drive")
PROJECT=Path("/content/drive/MyDrive/mediflow_Project")
ZIP_PATH=PROJECT/"MediFlow_Smartphone_WebSkin_Runtime_20261111.zip"
EXPECTED_SHA256="edc193fee3c25254ce947b1a9613ad8ec2cc4d0f90f3c4f283eff5d415ea3ad9"
MANIFEST_SHA256="ad2338eaa932fea3c209552f0c5e08b2cb31de7d542fbaa3e541ecfd74871c9f"
def digest(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
if not ZIP_PATH.is_file():raise FileNotFoundError("기존 공통 ZIP이 없습니다: "+str(ZIP_PATH))
print("기존 공통 ZIP 확인 중",flush=True)
if digest(ZIP_PATH)!=EXPECTED_SHA256:raise ValueError("기존 공통 ZIP 버전이 다릅니다. 11월12일 학습에 쓴 파일이 필요합니다.")
RUNTIME=Path("/content/mediflow_phone_test_ad2338eaa932")
patch=zipfile.ZipFile(io.BytesIO(base64.b64decode("UEsDBBQAAAAIAEcZbF30XhVTnAkAAPcmAAAUAAAAZXZhbHVhdGlvbl9wbGFuLmpzb27tWkuPHLcRvutXDPYsrUgWn/E5yClAgAS5BEKjSBZ325qZHnf3rCQbOiQR8gdiI5ccBATIyZcAOfoXxfJ/SPW8tmf2/ZDhONJhNVNNVn+sqq+qSM5XTyaTo8UU51V3isrYo19MjrwEoXLI2trIf0AW6UuI3oSSpbfBojApgwjBY8ouOAwJtck6K5Qm26OnW6Ws7Sv+zN+6dEozHLTPKNdl2ryqFqfNnKpSz3Fa9dT11ZlczeTRPXYvd3P5e25mWA/ajl5RrLqX/Pnp9lmaYtdRxw//sBGx8D//+veHd+93g1jy4Zt3P/zl/Q9ff7cn/Nu33//179//892e8P03H/78xz3JP7778KdvjzaCF7tXd82yTVTNcUZjbFVGXgD13dHhyHMbFx+ExwLZaDJKYpY6gUOto7EhAxQKKhJEH0O0Fn32omDREYPVwql8rrqeL5Z9tWimdXozaE4076mtui+W2NL5sIh9Oq26+ssBqt+JZ02maTey9QCWppR6yntSlmObTuueHy3b1XoXs5ORlYaJa+28xD3xFCNNq27WNP1pPT/hEeJY7I1oqVBLc7ZQwnmu2X5U1Xn1jmXkhY3EZ6ri91ZRVPyaimcooawISlUKjNeiEgGyFJn2odXzuq9xWn+Jfd2sIqnDM8qT7VonHJ/p5aKp5/3+xJWBhvEtdctp3z3fevn5dmq1tuHzM7V7thYdv6QWu0vUjbkGZE3IWQQMYIwEQ8TghQxOgCMvKVGwyjlXeBhlJJ8Dgoua/5akj3bK3z4Z//92w6TN+xaUxny6zpOXePF6D35c793Lc4/otY/ksT3/jPm3Xe8+JVu2/Go5y3lfz8axfVTqKT1SfB49fly+PQ+rHk9IXr6sRVs37WWLWk+qtkt5HrlOXAcbQkxRFTDWFSgmFjROoCrJUYqGPAoOL7RCCikxeGEhe8CcIWgruJZdAVvdB7a6ES1ywdQuhCiTzUXFkJyIzgAooYMUXG6zR4fEtRWTz4oLLQnlMyAXB5nzFWjhPmjhRrRZOsWlChwEn9HJVCxpkkEqlZQkjDmFRKqgECIV7wkYt3WK/yqiAleg1fdBq29EW4QPxienjZcyJ9DZk3eWnFYlsPNjksUG8HHAbXXMXqHyOjsSRqVAV6A1l6P9/KBw7KM1N6IlwThc5niwWVA03udodLAlWiJuAzgyXCgqaaaZQ4yFW69gSjZCl2So7NDuJZfUcN4dsua4O+KGiXDUP63bsGl9ctoPqf1ATq8XnLwpH8o559f9oXBthZVs0ySxG5vBFHsZ7uCN+73HqE/EjstT5uoxY8Mxkp4GJOdt4mrCNa3hULgue8lNs9YQ+2ZGfcvVq1nw63sORh4dm+UKA6cMNuwZFzTYnxjb4Y1z6rqqKYWbwL2+dPj3LBx7qYTVwWhpFHfRT/eec2UdfX+xp50d2nMMXVQq9xuqlSAIDQGEByu1UXCl0imVfu2Ni1jFsZJSDlq0k9YF6/3t0XYJp1S9vhXYG5S8ubOS875oFA6pmTZthQfKDrrR8ZJeXJgc7ze55/DlaK5O1mE31nCwEnmFhjhdcldfn8ywapva6vvhWGlZbxlWll21clpKZcT5P3sh9x0kg0+U/UTZnxJlWZGUTnq3jeLwozJYHAdzAMc8Aocle9J4YdiPUmkrg398Sq/r+P8bn2U4tkpLFQJzTWi27MchtPTCO2+Zhkoxt/2VSq8ntFYKBi1BA5gAzvzsCf1j12Bx7A8YLB+DwepYg7S8k2MP6qAg2KuV3pfB66b7E4OfHgwQzD4u2lIrA95I5z4x+lDJ/0yJfjaUQmWDgS0n1M+36X4ysu/RDOd1GS5Hzs8KXPJRsyG998B9HpmcQQuMHnV0xQuNgPwBeJw0QQSI1nJX6XIIDh343SXLoDYxF4cI1UKMxd1ysWjacej63Upv+vRktPwjJta8K007qzpabRykE27zjM5wulyd6lbnwxZtEzHW07ofonPnpt2lRMWjed5wnLw9Dl6fIo3hLzseyOo2B4frgQWnHW1HYXtCfbXKUN1q+MGAtbdaSk2bx6cndzqpWs5m2L45/rxr5pcf+wRSXmAoEixmYdAXLZIskT0sODcQojNKeptc0VEiGlCQhOdHw9PiLxxS3QXe5iy5qjPxh/46nNnyniJjyIkyKDTCl4IuM3pbosFgjVagionkEHWOkHXicCMZg/fo5INwbu7Otky4BuaDmfEQmOdByaSaLbCtGec1YJPygawhEIGCLzYaHY0wKoKGSFzbtAGTjYCk2fjBZW2LDX51TeKVKA+z6cG5+oB4Sj1dF6uomMFlOEUly8HoCGNkXNGA9UqCkFnqGECSA81FNyksyEYWxamkyyPgVbeBKYYjXQJrrCWrVHTcdYHVJiefs5LR5mKKJACQUmMJKIpyJRsnmHvex4fDhNvANEVYG5ISqmgOPaVLdlKBzCoHRUpxx5KGzoXFWucyXJExzWIMQmkoSjwcpr4NTB88O5JzdhIxZh8JjIxFooecrLPa6iCDwJz4W1GJsDidMzddLjtVhLoNzKvPzm/Mn7wrTo7DMBZQWSBmlByfwrjkoCAZ0MVYwGScQaUIhWBHc2hEXkVU4eJ1z13g3SV/uujAKGMisK+jZnOxVQNv75XxTibJWYm7SiFcxOCJ11RAKOUMJTCAFy967mTGn1T+vBLmXfOnMdxAhxCN88DxaRUNF30cAD6paKPIykofkmVWEzHc5MHrbDRwoQUvb0X0q2267SuuAcimCckQ89oDamJzJqM9sYwrfWL/YyqYpVWBtLQpa8nplXM91/oUOUoeBnB973QLiqNCNhACd9eBQSje2LCncwDAYnVETjecOoEKGMEc03YI14ScTY31Am5V26/mEHG/l5crUz7v+mW+jkV6+KkAcfJJsUQxXJd54QTYhDpxYNromdQqeYYeXElWZG+sLMEWo3mjFfYvybYdapd4bzqo/+2KJ5PfMUcmzXz65rPJad3xlrXmln30I4MJvV40w6Z6gvM84W0t453UvKFdDDvrOWtYzs+orUu9uzUbMvGQyaovlqx807b+ct390qTUrylPfr8L/8nu5xKTdaKe4LCBGVDxvnj4udTkN7/+1WeTTF1q67gZ1clnZvKq7k+bZT9p6bzx3UBgSzddzct5U50Srn6dUSxTQ0rBbYdH5gkKmYa0zTmUc3dCCCJyx5G2KpZzDqdZ3Q9d+KZ3H1rs82KhkYuX5sQBIXHfQkryDhm4d1QlcViz+yynjcEnAGSZjZmbmiE5c1MpJIysxXpH+eBxSf/2ydsn/wVQSwMEFAAAAAgARxlsXSGya+rvFQAAYEQAAC4AAABzcmMvbWVkaWZsb3dfZGF0YXNldHMvc21hcnRwaG9uZV9maW5hbF90ZXN0LnB5rTxrj9zGkd/3V/QROIQjj6hdPXy58c0BPssKHPghWHYO8GBAcDk9s/RySB7J2YcWC8jxOlAsG1EOlr32SYp8ZyfnQMApJ9lWLrkv/in3UcP9D6mq7mY3X6u1LwNbMyS7q6ur613FtSzrUhpf5RHLgp3T83jCQ5bNvTRPNuKIMz+eJ14aZHHE4L98g7M4DWZB5IXsSrxIfc7e4FnOsiQMchgR7jqWZa2sTNN4zlx3usgXKXddFsyTOM2ZF0Vx7uVBHGUrK+peOoMVMq6uZ776teFlG2Gwri7fBiTU7yT08mmcztV1trHIg1BdXQ2SaRBygUXi5QhFoXAZLsu1N3nqZeoiWsyTXeZlLErUrZxHWZxOw3gb7+dTubE5nwR40514uZfxPCv3N/ESsT93ijRycyQOTFVEO/n8LPdmfI2WTb0gCqLZU+b63hx24/IdQHrCJzhT3HrKvEk8B/iusTSOwOn4/b0n852Ep8GcR7RvfdUByGlCCGIFfTsNcu4maTxLeQYc88prF198+QobMtsCiuSBF1p9ZglKlb/Olr/Olb/Ol78uWL2VF1579eJLb7z02qsClB9yL8IBxPFuGMw2ciC3vgOb8JCk+k7Kp0GuL9+OgygHwK+/eOXNl99w33rpMgC2XoG9XsK9XinFyb1EXIEi477Os0WYZ+7Z1bPPrq2tnXWAa0FyViZ8yuL1t7mfuygA9pYXLnhvsMLgk3KQp0gJhpNteGcvPGvTI/yghDgTYONMzOoz4F8UQC/zg2B4yQszuAccnbubfDcbvpHimIyD/Hl5nGZD2+rjrgZWr+fwyAdlYPcIes/Z4DuTYAaIwx2B5Bac7HTXTRdRDidsy2+J6dyLgiky/5DYyEm5N3ERPzWOnWGW/OmqwQ4OsMSKIN0sAv7tEw/5OXB0EJVgRxYKeGaNHeCQeWbLVfETTMWKkjj0O4x9IDtqArV8n4D3euxvhuUCGgbR2gsyzn6GdHwxTePUtkjTFTe+OPr4l+zogw+KO39mR7ceFTdus+U31+Gq+OPhgFnsGQGagPFoi4dxwo8lA8fDErwPei0yqYDXMFeBGVl4wxqvyI2afIJPxHYqg11BB2usd9fc2ZOH94p7t4Qqf/Lw4OizDxs7kwjBoprfEP7IyvwNPvesMa5tlQIu+F1rQXdrzSonwtnmiyTktoBAVgcOk9AXQt411I+jSUAGRA7XwmxOkZjxkA7WBcoCjYMJEVliKhVDcxYopk1rPLKEZpKjt/m6m23CZct43N0ig2WAZ12xplgmAFsS54zkrm2ddMZzJFQeZwSgbUrvuHMr3v9i+bvrR+8/PvrVA1Z8fbv49/fKQyxu/W9x5xFb3vhi+dv3ljeuww9HHqJUI4pRqvIsDoPI3WdpHOeZRAElMuV+nJIkij0EUbLIXXE3M3kMZU0xvSl9CG8kxo8svLLG4z5TN1Co4Vy7ZBlB0KGr8U3e7uDvxw+Kzx8BT39avP81g6vl53fqxBGi24YJLpvBZvb2a5opiKaxpoXi4qZKOpYYCMQghbg8KSHE6JOSgVhj+fv/XH74+Kmaq9z4CO+NAX28MtmHnkreEXywFfDtzM7IL0Te2c76TPDRNA4nPJUkAQ/xIkfjQv7k+dVVwbMEA1ylyOfPMS8MGegVfzNB25rhufBgi7NgAr5EACQE5waeg5uTpt5uRl6nOBxcyJlvToIUyJTCaGXn+E4Aghpv0qXUZgQt34Xdmap0T6jZgUTewp3A1Qi+HTpLEgVwCuHs8dl4X4DLgquo6Q12cDMwLahM8JHU2+ANgN8J4wSuaAHQyw55zoXqF7A2PDIbdsjBWMAivT7BV/+eK/WxgOfQ9ipsl3lbQKCG5RHjK7xFI0GaJTmEzlNXJzOLd99b3vuSFf9zE7mqOPgC7EmX+qkKkWneJR6Kn5uS1CILJRWJd7vt+cmR/8XN4t132uSB+BsVgT+AIAFE2Zvo5afWnr/vRMkueFDzuZeQIh1aqdWjzfq4P22s9k3ye9GuTfuBEwf8xQ+Y4zmTfFfcg+XArHr5ubMEzUNohI5Dnh4QqfcDzunonfugDL/79ujwVnGjoRHN85IyT0vSzYjzCXEX8qfeV4+dYiXHwm8SifLrHPx/XvGtiNmcSZBtgvUDYyxp2XOmKefsH9QSz7C11bPnT506V7eErwsXSm7vhTj01tny89vFn+6zK1cuFncOWcuWnzz8+smDA9rqN9eKz/9wdOur6m7LQy5Xk6cNDjeGnF7ugM2M3DnHU+44fzr67WfgJ53g0JKHBzfodIf0b1UcahxCzwSbbAegc2RQ67wVJJfgW6rZHkZZXupvgHLsQ4TqTPhW4HPbOvPC5TcHq5bBFbhG0Fd6i0O4C8FhzsVR1URsDsfBhkb45tAtVwTEdkOwJAakZWylJkuNVfWrhK1I4jDwd6VKVJ9e5YpOYiTjs/EoQCNEaFRGZZzY0EQVtIaLtxGTcvXUizI8P3pi2taS/sq1rJ7DaG0wbmoSgVs5RWInAm6nXKtJqJK6Ci+IbmUoY0ADm2MEnYhwA1CvrhMDNhyyVVQbdoAy02N/C7a1vEn38KqUzuamVKYBVFG8rRTGv91cPnz83bfLh9eOPj0shemLa8s/HgBuBLavgergjQxzU03pVWkAaLUFWFwxrRrva+IJi9Uvr/cqmGuzNShtVr86QhqUAdvTUjpotyJ6QLve1pDlz4pLLfSj8InIfYGoIAOPksuIHBwiPwa1BV+hl2VcnYJ4yEjPexlRRk5oCbvEA20v7BpIM9SQY1vsiDkKQw54EmRTzKtwtbQDXphdAYfoRQo1UNGrxzz9R+DBlkUAph/GmVrEyRZz+/Qa6ApgIy+Pw+EaP32h9/TYR3Dn4fXi8eesuHezuH6IGl9ZtOL2vaNbt4uDP4Bxu/UVqvvi7uHy939a3jwsPrkOUUBrRCRQkueXeEEKoQ1maex4kYPSEoo97GtV0dccJ/1deQgg3imEwn4+fBXi4L50bulCe8GYApqLHCt6fxMB/oxWRIhCRmlV5k1z4E8Ro4GISudO8BY4EDCX52nga08YXRdgKJHRfYZZrosOTQlbOJk43ZXhidgjuqMCqgU/bYICs0EirkpaiYWasyQC1WlGLgNdVT21xV+VwWXTYdWzKl6ritXa3VZkOVPC9V6rISTttSVL0s51xb1rxd0vlzdvMyMbBOwm+FBkhdojy6orSSZd+ZAatT4GPpg9CfzNkIuMHZl4Ou4qcvLsh3VFo5xocYhjQzm3qAh5aFUDSqOCKYRZyCm2YGwBrwVEea6ZSVfFDd+bor+7dvTr28Wn16Rsn4SYSQoxIkD69GD5Hx+wJ//94MnDP4O83y9+fr/47KuB1ZehBtkaI/wzJF+iq7apGFYKMaZlUHQH9Wl4k+6tezkcA9JxNNY3mGbHkUV3XCMQzOMc4lgzxIOdiXmn0X6fOSOuSpOa5TwBDHMvzWsunBeB+7xaOWqaimrVkLDSzICjCJoAtEnu5rHy6gzbI9TVSCw1kEtK5MY9cjSlGdGUlCRwvATc44kJTVKxNCtUabF7yjnWNqlXjUphu+ixrDG5d/Bo1rRHox4TGatcVnVkptZoj9TgPvvuWwjelArcHwuWWz54gAnOvqQvwetp/dgiYoAxQAHx4BGSX+681y1q/08x0wrCETkNmeBoSWmYKvZpY8FPTuLUSzH/YSyBusnNFtNpsGOT4kdAVG3p6XCknEvRkG1tr1tCUeWguOf6OIBUqJCuumjl0K0Dt1yMUXsdiq8aSqD/wUGGSMbQ28c7iXrjhqfY6hyymtYfdBmKvrZqA3WGFbdPaB4dYnVzmlROcPt531+knr873JMQRz/y5K0fjfctAxiOfsXz0/j0pTVj9BxvudM1HK23qpVbi19aV2/SwwljsMDKtxGujtQWsgIH0S4OwiqszF2pa2cBLpz1/Gym6jVU3isfKwDJLt5AxkjCfMVI0VVcB0KkJYHXwrMQ5POQ9Kz1miyqYsnqZaNq98/c22SohTzQiz7HW1fyNAanqXrzp5haZDIAkzp5GszABu+Q0APG4J2uCyqBd3q2j49RgQ/ttfN9dgEF3tuFbQyxLpGR1oGIzYiBAP++JD5qbIjhbQTeZ7alzhwxUSdqYciMg3Ox42rwLlw5ACNKJFWVR3Qx/D3bYjYIMPiI4Ef+rCx+9Cyy1zRoqKsgDGjKwSWtRpWIvoPbb8axwuIABRpPRoKPDA9gRMtBfAu/6N64LbIaN0GJuG9oxVbzGe13SP+2PIRz2A4m+cbw7DHbXatO1HunfWc8d3dy8MMyu9ys5D709UWxerh2oWXabhjM0RqvOatrLY9BB4UgO2b/BAg6ujWCPLUpIZ+hPZ3GUU689+Pa81kaTGwvTDa84aoj7THwKald+NahrahU6Y4OJyF5mSTBcO3ZVVVwRLuE4RlM1YzcyXrHycsFU17+rs/OOc8eKzFqMSE1RowleLsiP5pv9KG0S45gJHi0Uw2vu9lUJ3UQyekiw8qsAFHPG9EBBHNyM8QIQHzuJUPrn9DVAepuzYNouIrf3s4Q/i/XpcPIFgmqSQDbAlfzimBzVrLA33eMrvAr+iICp14X9N2TjcdDscFs7sDxkQ+HZwGkjCbaBZUzm0klWiznO7kNtNmlU5I9DX0GLGv5YJR5ipQyL9q4nZh6kQiSTC3D0hqi9BzG4cMceAA43o/DxTwaSv+TT54TbhV4jygRsnJn1Vaoi42tQ+gaO/jg7aIM9WpC1CJIwuCKs3eRjYXxk/Sqx9K6kUQ+JvdL1uBq/hkAa/PP6uliOQSMzTaQt3z60mX34ouXXn7+jRcvmqnkquHBoVSeAVYF902g6aSzMF63rVNWvfQAGpeAB5mL33aP8hN0S2CNsGyZGwCEHD/bom+hj0Suga7JODrzidXGVzLlTP6gjdD7Yo2Uh6CbtzhENorKIPRuEmdAr17PpGjpZGr30ihuqkNbRLprRBU3wfGL0z6jzp8+Uy4UFXf77BQ6U/+yCMCNnCULUyF9HzhozL1Em97Qm69PPJYMqIfNTnqAfRaHWxwjqe+FYc9wEPF8hUuNJWg6merYKiuAt5SX5Tq9qLlYkxs0fOQJ83xKcD1mAq8P0wBaGKErNfjN7eW9L88Ud39dfPIeO/rVo+WHHy3f/4jJXgCZyBb3Md2w/OZg+dv7WBa/hX09XzVrYBiSEnFPjF4TNampQO6KO9dYJ6IQk35cfHITsLxefNyOkdFa1NGJJQ16dweRKFGr9oa2/o891aMF8U952BadNpbHxalbov9tII5/v8z2GTJAGgATwCLxgHo2RNc+2djNsJrvirJVZls/ufym1cj7tlT6gDrFwW0Gw9nyNw+O3r1W3D0o7hyw4uDe0bt3jm4dFgePi88+UtTS68ZgQOZgWVKygDocByTwEXgl9p4lnBS3HAubo1yc3JzUgCfsMiA5OungsjtAR0et/WFlSrVa/SdCtxf8zd6wY5uK7jxa/tdjpvNytcaio88+7KoU12JzWbcpVxWDpOLG0jGpOUw4C7GiDki5L2QWpeK1IanldOUA2lq1yCggtqR1Za7wYoooaGlk//eLf2X1EjJbPvy6+Pl9qz1zKOvXfpzsTnVBtq+wluYkpmKy6v91wBVBFqCWOLus25q4q5YZo0FQbqatTRIoNcIKSurIAHDIAtGeoppTRBkroMvSz6sSS+aa0Asdl6Qs91n1VOn2sc2ExCdH7zxa3j3E/rSHIJPXD5e/+WUX06iCu9lDVFLGbCIq2UVkJyvliDK1QX1a/Uqv1mK+jlxINYlK2lQEM9UUqeCPqSXbpPbE7P0ze0g2MaEnsjsIb7+DN1SuhnqnFGplJ5WBIJ2hyEsFvNp2gB9/ULtBB2L2dQ66+j2bUbFqSVITq21rAjvdfNMyv4yMYLLf8tzIcA9q8mgmv1tm1juB2zNzT28droLer1x19VhUR+piXMtZtJUERWnB7xvHOPLHinOPFbYfgJ5MZZSFiaEuQeBHNhPJIE0ULQhyGbaV++tqHKJ3Ipx1z9/k0cTB5gtsas0w6rGrGOv+PEOX69IfAapllKRi1zO14aqpdj3E0O5dvNp0CuvK2XB1lFAaK9Rx/Csi0tSP0r5IBSNMzInaMc3zF6ck0KAiovDbDGz7dNSBqiKCEvKmXDSlVXUVketEnYsGhWiMI1S2bkQQte5qmyIeLKUWaIaQmoxIef4k1Lr8yk+kr35GOMpP6W1WHy0gZTFI3gKosozf3G6tI6u9I6iKNuoErBApuTLyRxTTGCqjsV2pQxr3Sac07uqMWOORqXt0F1GTvKiTmtnSY3RUc7SkYvNBvSWqK7uKn4p9rDQ+mabSJOWQKFkCQSlQpygQKh+dTIHNfFAOIfbq2yrwxzOqaH31CsDA6P/XuyqpVLd1JfWMsTL1uqA4SVcJO5ysgflqQZkjNEZ3vNmg5nW9+GBAaNS16ri0vNEggyATj4qbXSdEzQk3VxDvPpQ1kAFZKjFgvy2U0NEQIScOK1MpJFm6ErGZcrUFBJ3uLi22rFCM28fj4bmSJh1TKMFmoCQ5EGNHymX1VE3UwtjHjyegfYbWIp+e/vHpLMAcV8S3sUYxtNqLpuqVvmxrpbxH6JGmybYErqkspPZqY+TTeLuqcEYNiRWqr6W4Ynh6LQ/lsaIxbXuMuVc3iCZ8p+1pmYvtHHJK9q5QvxJmi4PWEEbl0c3gpQpMG61m33lXGe3Eal8NphZT2VchKxRmIb9Nw3WWKdTn2GM0P80jNT/tVsT8HGNRzI9q6H36qI56nPnB8ArLMOkMqyGyJaTFzpifU7JXMI8xa2QfM3rc+kSlwlrK3VL3g6Kp5FzozmQh9J4e012XNhnL7ghP+l2hvPHORs1v1SGPiDMEpmcMz9BM0JsRUpb6Zxqv24Jqkll7TK4fs67Ip6vVKK+ul/zrZgq7GjqMfsTFfI4tIYKI5YBaG3CWe/kCDVr5Os2kpl0sbQwyaYZlRC/fWjDeY6jNJKtDjdtZuwEXKCi7u86nsczyfA+jTDB+QHCvuIqQlNlC2PyA6W4QvRHpDbt8h/sLMaxm1cW45tuJxw5sNeVV8p3Eo6DR8XQa+IEXur4H+gnoBGZ5AxV/Fw5ApSCNI0zjWq0ZE/0iPzzPp47rAg9TIc9tsVEich1IT/L4sdTLZtEbIscPTHbzDeWi0d8ucMQdNalNq1mzZIEvnrUqNZ3SNpPYDh6GyKbDV+4FIKGzXisA1BwzVBsnS8o3YNS4sKs1fgJnDw4WGdRhSwlU6GbZwtn1Qqtun3rrpcvY0WlAbabgZBHPGCNreejO22XRNc3IoVJ/gcJ5Pp0tkIqX6YkN0/00oJrA0HUnse+6x+h6rd3pDwOkQLt4gR0AUunBD7Fh/EXZElP/CmQcbzJxPYmFbZ0+rZIAfUadkpepRVjWVSbGhmEOdWAIKPSFcDIV2ywiGy+dUjPTlc5Yw4XCWF7KuiH9ViaTLmQFcWUlmDLXReRcl3prXJf+koNrqb8AgKRe+QtQSwMEFAAAAAgARxlsXT0i5HUGDwAAuyMAABUAAABydW50aW1lX21hbmlmZXN0Lmpzb26VmtlymzmShe/rKSp8PVGFfemXYQCJhMwxRSpIyq6ajn73+UCLkmx3FKkL2+LiP7GcPAugf//2+++fTvJZH9unf/3+6VHHdu4O3zZPnw973Zz1dN4cn/fn7aNuvtpP/7O+Pbc7PfHlf/OCl0c9Pe/Opz+/ad+cvmz3f550p3LWsXk8DN2d/vzqXj/7/tYfX/TY1hM+Fa8p1jFMbdXHaH1UNUONrdn4rMWqaE0u5zz5mo6mZdTmcw/8PSVcBrQmcHg+im4e2347GfIf/3s67Nfzs5QebMmlFG+r1ziGD6b10kLPs5jQfOMHz/dsrKb6npKvPo9ac8u+vD7/KH++Ls1o53ZSZrzZbPfb82bzx9Pfq5gbVWJOWZrvazpiqxipPTYNRlSy5ByaxJBcSb67Zq1m6dolRKlj/nOxNtrTuZ23h/1Gno9f9fRSVaP21my20ilvTA0jqmsyJDqbeGFFY3PN1+mmjSk6zyyrG2JaKbXdXXVu9213QcS1cuk1+FlabcVOtTYXac2EGVjHMGdqdpggMpUvDK9+7eqIbvihanu6u/JTky/tQVntqUfdi14HkFjQ6qLlr6ldTReqheKCHy12W5x0Ma7Unooky4LHZiU4U2SKHx8dwHb/8FI4+pjKsNGxxF5YAZOClpZzz5bN8E5iWlANGmaajHAkxmpNcBZMW7m78Onx8OU62z5sMJqcc/xkDNMtoqN2alimboafwfQCAvjOGMrsa8htuKzVtuTvL3pmse1L1QR8m/rYAapts0Q2mvVsIYNbTTpi7lNaLsY6ttlppcdWR4PoIOV+eH2vupHDrvWX2oMn5dST+pE1mSigKtG7M7QaZwpZRjKz0NihRLa1hu7UqC1SjPLOx2q7l6o5saCVXfUxRKPdW+eCyXV4IONiYz3qnKWaFpOwpSxJSbSTyzHV1OvHqvqXqrCb12Z0PRUK8fRHKQmgtSLGQ3a+S4SjfOpZlcbt3oAxF9nckWz+WNXwvo/jADslhzGTuxBw9rHMMpxhKMO14Y2ro6Uce+gFsPXigcSwyXSb5SPLTPc+nD+/1A0zuinMpGpMozHhALRDGFoh4J6TaU4jxRhU1jmCD6OHDL/M0Onlf64r7RGV2bTnh0fdfx/AS10ov5VpqjrjS2qtifMTRC1BaRCjsuvMzTkQVlSDDeqLwDDgusHhd9XVv866R7GuoIJ4hmW74KPoa5FuEkzBgrLbcZqRB3RpO9xYJ6sB9lxozrekgar2rppHndvrpjbDwkLBBob2MmqaTXXWEelVndn0jiDY0Ut0DjwxPs1L8vK02cV8gy3eF0QYHp6Pb3pUbM7wYV7/OCPgp9BHrqdcqw7omQZlPS10FtaMJZlhplgdcHe7QRivhb9uT287ykxaxEmgedOBJoWRFumHVnhqopdVaRaYSi3r63r10LBK7zHnJPch6bR93O7acXv++6Xqwo6kCcnXrK7G5BMTp32MIMcJ+JrVxDkCZwG9ggiEAsiTF0Yy7qv6Y8+oNas3eT4LDG66pAXn3g0jyLYFsbaNGBDYxE6suU/Pf+h8cxTtt2rux5ZX+ovi0aYm5Oi9wVNMnjmd84DKiGeNY+ni4rIeQ3vuxtdYbTSzDwgsO9Tv3rpHfToexrO8a1elQo0xlNYXA7dVYhicjmGqeCraF6FLAWKGF7FVdHDJS5yiy/OWzRDYCfRunra7w7V3ZDbIf4zcO1KCnPQcjZjMEGz3CV8Hgplcgy9dKerZdA/yhB0JPt2C02571VasrgS4zpTecGM4UgwTvgK3hKWqDV+sqZZFE9gZybjk9Qe8LQdww53K4fER0m3P45UUosUI2dQwm4y7agaieIoYmFEWWqfWBAFMOLj2IFnp4+m8OPA05JZXfCkIXLf7q2mZYkwWo6ATsbbT2CDLHQkmxesol15hYQ3THEqrSGWnxVnoITOCuyp+Oxy/rA+uy1rGcEFw+nQAMaIa4zAtBpPc5+hap1TVSiGTbOsm5xjqhP74LMR+g/pefqAxD8e/f+I+18krNHosQdg6yuUWkrCkiMiosSOpakFWmkCGsABmTbWQMByPXbpR+fDYiFBvgnqFayhp2dEpPbFqCjc0xATLUpPg/zqJpyRxCWdoDHkKo4rgOay4g5luNMgvVTeNHPD3aXuddPOWXEGHdE2oWrPotHNmthh8LtAC1gLFcZosFrXROghfBGi5V1jyBjX8Wr63s1zp0BkUMy/zxa7CR+SQBB8h41gia40UHCKZbNl9uhjrqFB/DN0GeMJ+eOrroyuyQ+6oXA42R9snmSva6SNxj20InaDnzAj0W0ObUCOb4F92qZNc+ezDpfWvJz1ul5e5zp0nGislGZkV8GZMzYqwTLgtczppgRZqlDDC9ATOiE0ls6U6kgcVHx3A9nDVPdMGSkeMhpx69KuOjyRMlLyH4kMkArcpMxJGsIhRoi81z6yYthBu9dcvhZ+wGbvtw+fXidPFzC7AnCRQUqfviCr8XHHNAw8JCgeMqTJxtsZiW23VuiwWWPQfBhz9fdg9v2s4tg/zCFsQn8lgZa0qbyRvTZut46RJxfA0MahAr9VZ8BmqOFdczs59fABft/rttd1o3WpGRaJMXM7VCJ6joH4YuVrIJugjpo/VsG42nkMgNWUx4jqL+Gjx85E33owAc/NTyb2rgUYk+hL9ZvXMcpjsLq1mhT+8Qt6KkZzW0UqzqfpZbxjZa3m2e/8O6vRVSQWCKQXH2nv2+HNsrLEZC9+JD+AOj+MQxbTOWOwEERh5W/Ekcl/R/Ztq1WVodLBavkEjFUCbErzBurL43RQMUKtEzYxmI2JMrEQtE2+Zag739faq90PcxaxlAcAkD0wGDl38Eug2E4areNNjLaXMhSisvElC3CSTNfKE0dlvqId+bbvnZbHe/AfhGjOD6RdhCqQclAQOSa6wsQQxOgwiddGx5COO5X5GT3AJNs8TFu8q+E6ncqH7PWxEhquuLcc2ySC5mXVekwVzi9eSJhM7BUtj0DPi5Vn1zlhvlHvlx83peXt+PTXpxCBHnooYQry4J6bjmdWvZMLOIhaDSItQ2zHxCvniMr0jqST0+QZ0Prft8fvB2DXngU/jhGg7rbFNQ19HF3OQqUCMLBZuvk/KFZJZbC4WESkzd0KX7eaOcg+Hw9jI8XA6/eDtgAfaKwW2I/aYOFydJGl4CotR/dAGReJSadZprSeO4X0QDJQDJdR7Jsrbp+3Dbvu02YHddrwy0mDKGGXBPIWFS9KdH3Qre4kfFvwIGRPwsOVCKkCFCCalY+KJBrdOwy6VkYDTTyneSHeyGm+GtE4cQ/cDdQU40IHD91uH/2IkzvviHTSNCC3Pg9/m/9w4tbhUPbXHKwHBJLB36BjKKjbg3fpgigZlwXh5nL/DQONrmbQrkYlhYg02NividFex7V+0ZmNX32sdm1Uc+Yf8OnpeJwSErQF8yM9jrgREPEph9OHAm5tYgtWjK49dzsE+VPjlnPVaWmv2+FWy+0LvmNpnIxygZJWU01LkDbgQmsTUwIWN5ce5Ez3twGh9qPT7lrUdgcswATNH3xuLns3lnMSEQqqzuI9EAKODEiMytBzBaRJXWCJW/57Cz0/yiqSRZdiS2LzhfEPbGnYtWzwSboWoR1LC3xrsROZDrcvR4ikT7ZOSnzdy/Lt6K91qu25t986WdUSam2IYIdiMcBJ1SZDrQNlimTtktEJsTKnTZXDTsM6QpPTW6fXPZ+WsD9tXMN5MARj1BdMSyWEJNjAEPCN4Zih/3bhMXXnErrsYr7ONfAPB3296rrKiEI11kVk052uBSid8UDv+9yLTQfOkXwomJoAVXfjONRCvc2XR/7nUU4PnCQEn3b0Jdl/3G3heAOJHxhJ0w6OTIylj9afLy4GZDK27FBIoJemxiov5W4Ab7yn5HqKDPo+W+eUJuY5evUHOLEt3uQOxLvSu3WUmT96mKRYtXRoUL2xuee/3bLeRz+14fs1apHHIh+VLVTF8IVhAEgcAupxAFNtEzMKpmbNVaLaaiGUhjGWMzI04/f7w5Qdtwd4aZJO01Ynz2JxB5mAUqeCuhlRHmM+9AJWy7kJY5ykMaDGUS2TxG2Vf7hc7cP382I5frlXV2fq9zxWXPwr2BwRJlNlSnbWTMdnJ6K2SwmbLhJ0UYmLLCd7uRl9eriNlp41E+bzdDT2+8h7LtcZfOuohypovzuvTBHKUFusxmIkwGdapIx2a1v0D2Ta2MjC7N8oywfP3y9X2BBsc21uP4p6R0HUIShtAtzgSR2qH7j3ugRxlMWmoD34N3SFS0jQlZoQe0xluMcK7yvvDWfvhcF1oYMmaEabU+gvXJZg3rANpxKXBsYNqLqFq1gyil3cL2sA6ozX5VoJ/VxdgP7Xj6/XdtKEWaNSmFchN6rYFK9bExCuMKMR/URZM6ISuxsoPOSI7AdtobqDqh7KHdWd9hVWsFqWwi5FoEOwvqiGOdOQHcKLBGlbJsROJ6Exqbz0GGn0iS+juDXp6V/b7rdLP6Q0Az64k5OLSulmDqwSaCODbKBgPfUEOjzzZffhMYCqCQAwZK3c/uH6KbQS1woaWSDvSlC2pixUKw5yVdSFKjqlV12VfUlY/+UDisAAN3gDtN+q+Xe//d4eohaVjP+06S8QRQxaYBgJjaeRUKzUrwkPkcOAsJNtD6ybmEVJcR2c3bsZfq/8Xnm7E/4hfgaEwwKHVzLyCXZfU66orGAAdlih20nP0nVS5jl0FFxtJKfcWfnzYdLthb68xwFf4cZ2HT0wajbRiR85+BYK+7mPx/Z52M8pYIuJg7IiVVQ/SjKk3ePqHuu/TB84okG5qCfB+npY8YzEZ9JWBWtb909RE7BGSCJpUsene2zLxFSs03LvL306jXc1TaEMzgRVq8H2xoE5aqUCPDsDCHV5bF4tvq80sRpsi0xaMlqWpy2sTv6XFzdOOp19/gwMoJlOhXqR8XYtizsy6ge1VCxgaLUSII2H0TXWg2MVUYEr6dvjVU3c3zC+/3lCYGmisZMWKoRHUaNkLFg51s1RKJsW6yGp6r0sKgdKCTSQ+Gus/Ufg/l1+YWWZl8380xekzgS9dOmKIrX4qmZt8HQPOIcDB69zSk6cUPoJ8yGqGcIGBWdo6I5YsauML9dNv//nt/wFQSwECFAAUAAAACABHGWxd9F4VU5wJAAD3JgAAFAAAAAAAAAAAAAAAgAEAAAAAZXZhbHVhdGlvbl9wbGFuLmpzb25QSwECFAAUAAAACABHGWxdIbJr6u8VAABgRAAALgAAAAAAAAAAAAAAgAHOCQAAc3JjL21lZGlmbG93X2RhdGFzZXRzL3NtYXJ0cGhvbmVfZmluYWxfdGVzdC5weVBLAQIUABQAAAAIAEcZbF09IuR1Bg8AALsjAAAVAAAAAAAAAAAAAACAAQkgAABydW50aW1lX21hbmlmZXN0Lmpzb25QSwUGAAAAAAMAAwDhAAAAQi8AAAAA")))
frozen=json.loads(patch.read("runtime_manifest.json"))
allowed={*frozen['files'],'runtime_manifest.json'}
if hashlib.sha256(patch.read("runtime_manifest.json")).hexdigest()!=MANIFEST_SHA256:raise ValueError("노트북 내 실행 기록 불일치")
RUNTIME.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as base:
    for name in allowed:
        part=PurePosixPath(name)
        if part.is_absolute() or '..' in part.parts or '\\' in name or ':' in name:raise ValueError("잘못된 런타임 경로")
        path=RUNTIME.joinpath(*part.parts)
        expected=MANIFEST_SHA256 if name=='runtime_manifest.json' else frozen['files'][name]
        if path.exists():
            if digest(path)!=expected:raise ValueError("기존 5차 런타임 파일이 다릅니다: "+name)
            continue
        content=patch.read(name) if name in patch.namelist() else base.read(name)
        if hashlib.sha256(content).hexdigest()!=expected:raise ValueError("실행 파일 해시 불일치: "+name)
        path.parent.mkdir(parents=True,exist_ok=True)
        with path.open('xb') as f:f.write(content)
patch.close()
PREVIOUS=Path("/content/drive/MyDrive/MediFlow_smartphone_runs/edc193fee3c2")
if not PREVIOUS.is_dir():raise FileNotFoundError("이전 결과 폴더를 확인하세요: "+str(PREVIOUS))
OUTPUT=Path("/content/drive/MyDrive/MediFlow_smartphone_runs/test_ad2338eaa932")
LOCAL=Path("/content/mediflow_phone_test_work_ad2338eaa932")
OUTPUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(exist_ok=True)
print("최종 Test 결과 저장 위치:",OUTPUT)
JOINT=Path("/content/drive/MyDrive/MediFlow_smartphone_runs/joint_a3d4036b9f90")
if not JOINT.is_dir():raise FileNotFoundError("추가 5차 결과가 없습니다: "+str(JOINT))

In [ ]:
# Colab 기본 TF 패키지를 바꾸지 않는 별도 실행 환경입니다.
VENV = Path("/content/mediflow_phone_env")
PYTHON = VENV/"bin/python"
requirements = ["tensorflow[and-cuda]==2.20.0", "keras==3.13.2", "numpy>=1.26,<3",
                "pillow>=10,<13", "opencv-python-headless>=4.10,<5",
                "matplotlib>=3.8,<4", "tqdm>=4.66,<5"]
probe = "import importlib.metadata as m; assert m.version('tensorflow')=='2.20.0'; assert m.version('keras')=='3.13.2'; import cv2,matplotlib,PIL,tqdm"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode or not gpu.stdout.strip():
    raise RuntimeError("Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.")
print("GPU:",gpu.stdout.strip())
ready = PYTHON.exists() and subprocess.run([str(PYTHON), "-c", probe], capture_output=True).returncode == 0
if not ready:
    pip_ready = PYTHON.exists() and subprocess.run(
        [str(PYTHON), "-m", "pip", "--version"], capture_output=True).returncode == 0
    if not pip_ready:
        created = subprocess.run([sys.executable, "-m", "venv", str(VENV)], capture_output=True, text=True)
        if created.returncode:
            # Some Colab images omit ensurepip. Bootstrap only the environment creator.
            print("기본 venv 준비를 보완합니다.",flush=True)
            subprocess.run([sys.executable,"-m","pip","install","-q","virtualenv>=20,<21"],check=True)
            subprocess.run([sys.executable,"-m","virtualenv",str(VENV)],check=True)
    subprocess.run([str(PYTHON), "-m", "pip", "install", *requirements], check=True)
ENV = {**os.environ, "PYTHONPATH":str(RUNTIME/"src"), "PYTHONUNBUFFERED":"1",
       "TF_NUM_INTEROP_THREADS":"2", "TF_NUM_INTRAOP_THREADS":"4",
       "TF_CPP_MIN_LOG_LEVEL":"2", "KERAS_BACKEND":"tensorflow",
       "MPLBACKEND":"Agg", "PYTHONDONTWRITEBYTECODE":"1"}
ENV.pop("TF_USE_LEGACY_KERAS", None)
source_code = "from pathlib import Path; from mediflow_datasets.domain_adaptation_batch import find_source; import json; r=Path("+repr(str(RUNTIME))+"); s=json.loads((r/'evaluation_plan.json').read_text()); print(find_source(Path("+repr(str(PROJECT/"datasets"))+"),s['plan']['task']))"
found = subprocess.run([str(PYTHON), "-c", source_code], env=ENV, capture_output=True, text=True, check=True)
SOURCE = Path(found.stdout.strip().splitlines()[-1])
if not SOURCE.is_file(): raise FileNotFoundError("기존 web_skin Source ZIP을 확인하세요.")
print("기존 Source 데이터:", SOURCE)

In [ ]:
import fcntl, uuid
from datetime import datetime, timezone
log_path=OUTPUT/("test_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]+".log")
command=[str(PYTHON),"-m","mediflow_datasets.smartphone_final_test",
         "--runtime",str(RUNTIME),"--source",str(SOURCE),"--previous",str(PREVIOUS),
         "--joint",str(JOINT),"--output",str(OUTPUT),"--local",str(LOCAL)]
print("실행 로그:",log_path,flush=True)
with (LOCAL/"execution.lock").open("a") as lock:
    try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:raise RuntimeError("같은 Test가 이미 실행 중입니다.") from None
    process=None
    try:
        with log_path.open("x",encoding="utf-8") as log:
            process=subprocess.Popen(command,env=ENV,stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end="",flush=True);log.write(line);log.flush()
            if process.wait():raise RuntimeError("최종 Test 미완료: 위 실제 로그를 확인하고 같은 노트북을 다시 실행하세요.")
    finally:
        if process is not None:
            if process.poll() is None:process.terminate();process.wait()
            process.stdout.close()

In [ ]:
from IPython.display import display, Image
summary=json.loads((OUTPUT/"summary.json").read_text())
if summary.get("status")!="completed":raise RuntimeError("최종 Test가 완료되지 않았습니다.")
print(json.dumps(summary,ensure_ascii=False,indent=2))
display(Image(filename=str(OUTPUT/"plots/test_comparison.png")))
for name in ("initial","stage4","stage5"):
    display(Image(filename=str(OUTPUT/"plots"/(name+"_confusion_matrices.png"))))
RESULT_ZIP=OUTPUT/"MediFlow_Smartphone_Final_Test_Results_20261112.zip"
if not RESULT_ZIP.is_file():raise FileNotFoundError("결과 ZIP이 없습니다: "+str(RESULT_ZIP))
print("Drive 결과 ZIP:",RESULT_ZIP)
print("ZIP SHA-256:",digest(RESULT_ZIP))
from google.colab import files
files.download(str(RESULT_ZIP))